# Preprocessing Raw Gaze Data

## What you will learn in this tutorial:

* how to transform pixel coordinates into degrees of visual angle
* how to transform positional data into velocity data
* how to compute velocity and acceleration in pixel units

## Preparations

We import `pymovements` as the alias `pm` for convenience.

In [ ]:
import numpy as np

import pymovements as pm

Let's start by downloading our {ref}`ToyDataset` and loading in its data:

In [ ]:
dataset = pm.Dataset('ToyDataset', path='data/ToyDataset')
dataset.download()
dataset.load()

We can verify that all files have been loaded in by checking the `fileinfo` attribute:

In [ ]:
dataset.fileinfo

Now let's inspect our gaze dataframe:

In [ ]:
dataset.gaze[0]

Apart from some trial identifier columns we see the columns `time` and `pixel`.

## Preprocessing

We now want to transform these pixel position coordinates into coordinates in degrees of visual angle.
This is simply done by:

In [ ]:
dataset.pix2deg()

dataset.gaze[0]

The processed result has been added as a new column named `position` to our gaze dataframe.

Additionally, we would like to have velocity data available too.
We have four different methods available:

* `preceding`: this will just take the single preceding sample into account for velocity calculation. Most noisy variant.
* `neighbors`: this will take the neighboring samples into account for velocity calculation. A bit less noisy.
* `smooth`: this will increase the neighboring samples to two on each side. You can get a smooth conversion this way.
* `savitzky_golay`: this is using the *Savitzky-Golay* differentiation filter for conversion. You can specify additional parameters like `window_length` and `degree`. Depending on your parameters, this will lead to the best results.

Let's use the `fivepoint` method first:

In [ ]:
dataset.pos2vel(method='fivepoint')

dataset.gaze[0]

The processed result has been added as a new column named `velocity` to our gaze dataframe.

We can also use the *Savitzky-Golay* differentiation filter with some additional parameters like this:

In [ ]:
dataset.pos2vel(method='savitzky_golay', degree=2, window_length=7)

dataset.gaze[0]

This has overwritten our velocity columns.
As we see, the values in the velocity columns are slightly different.

### Velocity and acceleration in pixel units

To keep the derivatives in pixel units, select the `pixel` column with `position_column='pixel'`. For evenly spaced samples, set the experiment's sampling rate in Hz. The results are in pixels per second (px/s) and pixels per second squared (px/s²).

This example follows two quadratic pixel coordinates sampled at 100 Hz. At the center sample, 40 ms into the signal, the expected velocity is `[80, 160]` px/s and the expected acceleration is `[2000, 4000]` px/s². The center sample uses a full five-sample window without edge padding.

In [ ]:
sampling_rate = 100
time = np.arange(9) / sampling_rate
x = 1000 * time**2
pixel_gaze = pm.gaze.from_numpy(
    pixel=np.vstack((x, 2 * x)),
    time=time,
    time_unit='s',
    experiment=pm.Experiment(sampling_rate=sampling_rate),
)
pixel_gaze.pos2vel(
    method='savitzky_golay', degree=2, window_length=5,
    position_column='pixel', velocity_column='velocity_px',
)
pixel_gaze.transform(
    'pos2acc', degree=2, window_length=5,
    position_column='pixel', acceleration_column='acceleration_px',
)

center = len(time) // 2
sample = pixel_gaze.samples.row(center, named=True)
np.testing.assert_allclose(sample['velocity_px'], [80, 160])
np.testing.assert_allclose(sample['acceleration_px'], [2000, 4000])

pixel_gaze.samples.select('time', 'velocity_px', 'acceleration_px').slice(center, 1)

## What you have learned in this tutorial:

* transforming pixel coordinates into degrees of visual angle by using {py:meth}`Dataset.pix2deg() <pymovements.Dataset.pix2deg>`
* transforming positional data into velocity data by using {py:meth}`Dataset.pos2vel() <pymovements.Dataset.pos2vel>`
* passing additional keyword arguments when using the *Savitzky-Golay* differentiation filter
* computing velocity and acceleration from the `pixel` column